In [1]:
import pandas as pd
import numpy as np
from os import getenv
from joblib import load
from datetime import datetime, timedelta
from utils import get_stocks, get_candles, calculate_indicators_for_prediction_modul

In [2]:
tickers , dv, model = load('stocks_analyser_tickers_dv_with_model.joblib')

In [3]:
# выгружаю токен для т-инвестиций
TOKEN = getenv("t_bank_token")
# P.S: если захотите воспользоваться этим модулем, вам придётся создать свой токен на сайте т-банка
# и записать его в эту переменную в таком формате: TOKEN = "ваш_токен"

In [4]:
# получаем данные по акциям
stocks = await get_stocks(TOKEN)

In [5]:
# создаем диапозон дат за 70 дней
end_date = datetime.now()
start_date = end_date - timedelta(days=70)

# выгружаем свечи в этом дипазоне
candles_df = await get_candles(TOKEN, stocks, start_date, end_date)
candles_df = candles_df.sort_values(['ticker', 'date']).reset_index(drop=True)

Запрашиваем данные с 2026-05-05 по 2026-07-14
Всего тикеров: 153
Успешно добавлен VSMO в результат
Успешно добавлен UNAC в результат
Успешно добавлен CNRU в результат
Успешно добавлен VKCO в результат
Успешно добавлен MGNT в результат
Успешно добавлен KZOSP в результат
Успешно добавлен WUSH в результат
Успешно добавлен UGLD в результат
Успешно добавлен SELG в результат
Успешно добавлен PRFN в результат
Успешно добавлен CARM в результат
Успешно добавлен RUAL в результат
Успешно добавлен NKHP в результат
Успешно добавлен ALRS в результат
Успешно добавлен DIAS в результат
Успешно добавлен TATN в результат
Успешно добавлен RTKM в результат
Успешно добавлен PRMD в результат
Успешно добавлен TGKN в результат
Успешно добавлен TRNFP в результат
Успешно добавлен DOMRF в результат
Успешно добавлен RAGR в результат
Успешно добавлен GCHE в результат
Успешно добавлен UWGN в результат
Успешно добавлен SNGSP в результат
Успешно добавлен FIXR в результат
Успешно добавлен NKNC в результат
Успешно добав

In [6]:
# добавляем индикаторы
candles_df = calculate_indicators_for_prediction_modul(candles_df)

In [7]:
# удаляем бесконечности и NaN
candles_df.replace([np.inf, -np.inf], np.nan, inplace=True)
candles_df.dropna(inplace=True)
candles_df = candles_df[candles_df.ticker.isin(tickers)]
candles_df

,ticker,date,open,high,low,close,volume,atr,obv,stoch_k,...,volatility_20,volatility_60,vol_ratio_20_60,momentum,range_norm,range_ratio,open_close_ratio,volume_spike,ad_line,return
69,ABRD,2026-05-07,144.60,145.20,144.00,144.4,589,6.340910,-2.319250e+05,60.282813,...,0.637654,0.370285,1.722063,3.865096,0.333333,1.008333,1.001385,0,-2.314140e+05,-0.001383
70,ABRD,2026-05-08,143.80,144.40,143.60,144.2,1118,6.077057,-2.330430e+05,71.222008,...,0.637833,0.370306,1.722448,3.837147,0.750000,1.005571,0.997226,0,-2.308550e+05,-0.001385
71,ABRD,2026-05-11,144.80,145.60,143.80,145.0,556,5.873388,-2.324870e+05,81.077797,...,0.638318,0.370292,1.723823,1.001381,0.666667,1.012517,0.998621,0,-2.306696e+05,0.005548
72,ABRD,2026-05-12,144.40,146.00,144.00,145.0,2484,5.688941,-2.324870e+05,89.954638,...,0.638476,0.370292,1.724251,1.002766,0.500000,1.013889,0.995862,0,-2.306696e+05,0.000000
73,ABRD,2026-05-13,144.80,146.00,144.60,144.8,757,5.484706,-2.332440e+05,98.463389,...,0.638575,0.370274,1.724600,1.002770,0.142857,1.009682,1.000000,0,-2.312104e+05,-0.001379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10189,ZAYM,2026-07-10,130.35,130.35,128.25,128.9,4287,15.715868,-3.396062e+09,8.022800,...,0.020263,0.015681,1.292190,0.990776,0.309524,1.016374,1.011249,0,-1.486266e+09,0.000388
10190,ZAYM,2026-07-11,128.80,129.00,127.80,128.4,2264,15.024636,-3.396064e+09,8.725244,...,0.020189,0.015682,1.287418,0.994578,0.500000,1.009390,1.003115,0,-1.486266e+09,-0.003879
10191,ZAYM,2026-07-12,128.95,129.30,128.50,129.3,373,14.352034,-3.396064e+09,9.674202,...,0.020357,0.015686,1.297792,1.003103,1.000000,1.006226,0.997293,0,-1.486265e+09,0.007009
10192,ZAYM,2026-07-13,129.35,130.65,129.00,130.5,5752,13.747175,-3.396058e+09,12.213010,...,0.020365,0.012572,1.619925,1.012806,0.909091,1.012791,0.991188,0,-1.486261e+09,0.009281


In [8]:
# проводим ручную фильтрацию по техническим индиакторам для тестовой выборки
candles_df = candles_df[
    (candles_df.sma10_vs_sma40 > 1.02) &
    (candles_df['hist'] > candles_df.signal) &
    (candles_df.stoch_k > candles_df.stoch_d) &
    (candles_df['volume_ratio'] > 1.2) &
    (candles_df['volume_spike'] == 1) &
    (candles_df['ema_12'] > candles_df['ema_26']) &
    (candles_df['price_vs_sma_10'] > 1.01)
    ]

In [9]:
# убираем колонки, которые нельзя использовать в прогнозировании
features_col = [col for col in candles_df.columns if col not in ['index', 'open', 'high', 'low', 'close', 'volume','date']]

In [10]:
# формируем матрицу признаков
candles_dict = candles_df[features_col].to_dict(orient='records')
X = dv.transform(candles_dict)

In [33]:
# формирование предсказаний
proba = model.predict_proba(X)[:, 1] > 0.6

In [34]:
# Если есть свежие прогнозы, выводим их на экран
predictions = candles_df[proba & (candles_df.date == candles_df.date.max())].T

if (candles_df.date.max() == datetime.now()) & (len(predictions.T) > 0):
    display(predictions)
else:
    print("Нет прогнозов на эту дату")

Нет прогнозов на эту дату
